Comparing our Kinase-Substrate mapping with the one from KRSA.

Mappings from KRSA are downloaded from: https://github.com/CogDisResLab/KRSA/tree/devel

1. Loading the KRSA mappings

In [1]:
import pyreadr
from csv import writer
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
from gprofiler import GProfiler
import sys
from collections import defaultdict
from statsmodels.stats.multitest import multipletests

In [20]:
# .Rda Datei laden
mapping_ptk = pyreadr.read_r('../data/benchmarking/testing_kinase_substrate_mapping/KRSA_uka_mapping_PTK_PamChip_86402_v1.Rda')
mapping_stk = pyreadr.read_r('../data/benchmarking/testing_kinase_substrate_mapping/KRSA_uka_mapping_STK_PamChip_87102_v1.Rda')

# DataFrame extrahieren
df_mapping_ptk = mapping_ptk['KRSA_uka_mapping_PTK_PamChip_86402_v1']  # Ersetze 'object_name' mit dem tatsächlichen Namen
df_mapping_stk = mapping_stk['KRSA_uka_mapping_STK_PamChip_87102_v1']

In [37]:
print(df_mapping_ptk.head(n=25))

#P00519
#P00533
#P12931

                    ID PepProtein_UniprotName PepProtein_UniprotID  \
0           41_654_666                     41               P11171   
1           41_654_666                     41               P11171   
2           41_654_666                     41               P11171   
3         AKT1_170_182                   AKT1               P31749   
4   AKT1_309_321_C310S                   AKT1               P31749   
5   AKT1_309_321_C310S                   AKT1               P31749   
6   AKT1_309_321_C310S                   AKT1               P31749   
7   AKT1_309_321_C310S                   AKT1               P31749   
8   AKT1_309_321_C310S                   KPCD               Q05655   
9   AKT1_309_321_C310S                  KPCD2               Q9BZL6   
10  AKT1_309_321_C310S                   PLK1               P53350   
11        AKT1_320_332                   AKT1               P31749   
12        AKT1_320_332                   AKT1               P31749   
13        AKT1_320_3

In [22]:
# Oder als DataFrame:
def create_kinase_substrate_dataframe(df_mapping_ptk, df_mapping_stk):
    """
    Erstellt einen DataFrame mit Kinase-Substrat Paaren.
    """
    # Beide DataFrames zusammenführen
    df_combined = pd.concat([df_mapping_ptk, df_mapping_stk], ignore_index=True)
    
    # NaN-Werte herausfiltern und nur relevante Spalten
    df_pairs = df_combined[['ID', 'Kinase_UniprotID']].dropna()
    
    # Umbenennen für Klarheit
    df_pairs.columns = ['Peptide_ID', 'Kinase_UniprotID']
    
    # Duplikate entfernen (falls gewünscht)
    df_pairs = df_pairs.drop_duplicates()
    
    print(f"Total Kinase-Substrat Paare: {len(df_pairs)}")
    
    return df_pairs

In [38]:
df_pairs_KRSA = create_kinase_substrate_dataframe(df_mapping_ptk, df_mapping_stk)
print(df_pairs_KRSA.head(25))

Total Kinase-Substrat Paare: 1212
            Peptide_ID Kinase_UniprotID
0           41_654_666           P00533
3         AKT1_170_182           Q07912
4   AKT1_309_321_C310S           P07949
5   AKT1_309_321_C310S           P12931
6   AKT1_309_321_C310S           P31749
7   AKT1_309_321_C310S           Q13882
8   AKT1_309_321_C310S           P06239
9   AKT1_309_321_C310S           P00519
11        AKT1_320_332           P12931
12        AKT1_320_332           P31749
13        AKT1_320_332           Q13882
14         ANXA1_14_26           P00519
16         ANXA1_14_26           P00533
19         ANXA1_14_26           P12931
22         ANXA2_17_29           P08069
23         ANXA2_17_29           P12931
27        ARAF_297_307           P12931
31        ARAF_297_307           O60674
40         CBL_693_705           P00519
41         CBL_693_705           P00533
42         CBL_693_705           P06213
43         CBL_693_705           P06239
44         CBL_693_705           P06241
46    

convert kinase names into uniprot ids:

2. Loading our mappings based on the PTM data and the BLAST results

In [24]:
def load_and_merge_ptm_data(df_phosphonetworks_path, df_iptmnet_path, df_reactome_path, df_phosphoelm_path, df_uniprot_path, df_phosphosite_path):
        """
        Load and merge the various PTM datasets. This function concatenates all PTM data into one large DataFrame.
        """

        ptm_paths = {
            'PhosphoNetworks': df_phosphonetworks_path,
            'iPTMnet': df_iptmnet_path,
            'Reactome': df_reactome_path,
            'PhosphoELM': df_phosphoelm_path,
            'UniProt': df_uniprot_path,
            'PhosphoSitePlus': df_phosphosite_path
        }
        
        ptm_dfs = []
        
        for db_name, path in ptm_paths.items():
            try:
                df = pd.read_csv(path, sep=',')
                df['source_database'] = db_name
                
                if 'uniprot_id_clean' in df.columns:
                    df['uniprot_id'] = df['uniprot_id_clean']
                
                if 'uniprot_id' in df.columns:
                    df['uniprot_id'] = (df['uniprot_id']
                                    .fillna('')
                                    .astype(str)
                                    .str.split('-')
                                    .str[0])
                
                if 'ptm_enzyme' in df.columns:
                    df['ptm_enzyme'] = (df['ptm_enzyme']
                                    .fillna('')
                                    .astype(str)
                                    .str.split('-')
                                    .str[0])
                
                ptm_dfs.append(df)
                print(f"     Successfully loaded PTM database: {db_name} with {len(df)} entries.")
            except FileNotFoundError:
                raise FileNotFoundError(f"PTM database file for {db_name} not found at path: {path}")
            except Exception as e:
                raise ValueError(f"Error loading PTM database {db_name} from {path}: {str(e)}")
        
        df_ptm_merged = pd.concat(ptm_dfs, ignore_index=True)

        # Print statistics about unique enzyme-substrate pairs
        total_entries = len(df_ptm_merged)
        unique_pairs = df_ptm_merged[['ptm_enzyme', 'uniprot_id']].drop_duplicates()
        num_unique_pairs = len(unique_pairs)
        print(f"     Total unique kinase - substrate pairs: {num_unique_pairs} out of {total_entries} total entries")

        return unique_pairs

In [25]:
def map_peptides_to_kinases_dataframe(df_peptide_enrichment, df_ptm, df_BLAST):
    """
    Map peptides to kinases via PTM databases and BLAST alignments.
    Returns DataFrame with individual kinase-substrate pairs.
    """
    # Create a dictionary to store the substrates for each kinase
    kinase_proteins_dict = defaultdict(set)
    for _, row in df_ptm.iterrows():
        kinase = row['ptm_enzyme']
        substrate = row['uniprot_id']
        kinase_proteins_dict[kinase].add(substrate)

    # Match kinases to the peptide data
    peptide_to_proteins = df_BLAST.groupby('source_peptide_id')['subject_uniprot_id'].apply(list).to_dict()
    
    # Store all kinase-substrate pairs
    pairs = []
    
    for _, peptide_row in df_peptide_enrichment.iterrows():
        peptide_id = peptide_row['ID']
        
        if peptide_id not in peptide_to_proteins:
            continue
        
        matched_proteins = peptide_to_proteins[peptide_id]
        
        for kinase, substrates in kinase_proteins_dict.items():
            if any(protein in substrates for protein in matched_proteins):
                pairs.append({'Peptide_ID': peptide_id, 'Kinase_UniprotID': kinase})
    
    # Convert to DataFrame
    df_pairs = pd.DataFrame(pairs)
    
    print(f"     Total kinase-substrate pairs: {len(df_pairs)}")
    print(f"     Unique peptides: {df_pairs['Peptide_ID'].nunique()}")
    print(f"     Unique kinases: {df_pairs['Kinase_UniprotID'].nunique()}")
    
    return df_pairs

In [26]:
def load_BLAST(df_peptide_enrichment, df_blast_path):
        """
        Function to load BLAST results from a CSV file.
        
        FIX: groupby 'source_peptide_id' instead of 'source_uniprot_id' to keep
        the best BLAST hit per peptide (not per protein).
        """

        BLAST_threshold = 80

        df_BLAST = pd.read_csv(df_blast_path)

        df_BLAST.rename(columns={'source_uniprot_id': 'source_peptide_id'}, inplace=True)

        df_BLAST_merged = df_BLAST.merge(
            df_peptide_enrichment,
            left_on='source_peptide_id',
            right_on='ID',
            how='left'
        )

        df_BLAST_merged.rename(columns={'PepProtein_UniprotID': 'source_uniprot_id'}, inplace=True)
        
        df_BLAST_merged['subject_uniprot_id'] = df_BLAST_merged['subject_id'].str.split('.').str[0]

        # FIX 1: groupby source_peptide_id (not source_uniprot_id)
        # This keeps the best BLAST hit per peptide, not per protein.
        # Multiple peptides from the same protein (e.g. AKT1_170_182, AKT1_309_321)
        # each get their own best BLAST match.
        df_BLAST_filtered = (
            df_BLAST_merged
            .query(f'percent_positives > {BLAST_threshold}')
        )

        num_connections = len(df_BLAST_filtered)
        print(f"     BLAST data contains {num_connections} connections between peptides and proteins in the human proteome (matches with >{BLAST_threshold}% positives)")
    
        return df_BLAST_filtered

In [27]:
path_file_enrichment_peptides=Path('../data/external/PamGene/enrichment_peptides.csv')

df_phosphonetworks_path = Path('../data/processed/PTM_Data/PTM_PhosphoNetworks.txt')
df_iptmnet_path = Path('../data/processed/PTM_Data/PTM_iPTMnet.txt')
df_reactome_path = Path('../data/processed/PTM_Data/PTM_Reactome.txt')
df_phosphoelm_path = Path('../data/processed/PTM_Data/PTM_PhosphoELM.txt')
df_uniprot_path = Path('../data/processed/PTM_Data/PTM_UniProt.txt')
df_phosphosite_path = Path('../data/processed/PTM_Data/PTM_PhosphoSitePlus.txt')


df_blast_path = Path('../data/external/BLAST/blast_results_incremental.csv')

In [28]:
df_peptide_enrichment = pd.read_csv(path_file_enrichment_peptides)

df_BLAST = load_BLAST(df_peptide_enrichment=df_peptide_enrichment, df_blast_path=df_blast_path)

df_ptm = load_and_merge_ptm_data(
    df_phosphonetworks_path=df_phosphonetworks_path,
    df_iptmnet_path=df_iptmnet_path,
    df_reactome_path=df_reactome_path,
    df_phosphoelm_path=df_phosphoelm_path,
    df_uniprot_path=df_uniprot_path,
    df_phosphosite_path=df_phosphosite_path
)

     BLAST data contains 1540 connections between peptides and proteins in the human proteome (matches with >80% positives)
     Successfully loaded PTM database: PhosphoNetworks with 4235 entries.
     Successfully loaded PTM database: iPTMnet with 4920 entries.
     Successfully loaded PTM database: Reactome with 5109 entries.
     Successfully loaded PTM database: PhosphoELM with 1514 entries.
     Successfully loaded PTM database: UniProt with 7452 entries.
     Successfully loaded PTM database: PhosphoSitePlus with 815 entries.
     Total unique kinase - substrate pairs: 19589 out of 24045 total entries


In [29]:
df_pairs_pyGamG = map_peptides_to_kinases_dataframe(df_peptide_enrichment, df_ptm, df_BLAST)
print(df_pairs_pyGamG.head(10))

     Total kinase-substrate pairs: 4851
     Unique peptides: 256
     Unique kinases: 497
    Peptide_ID Kinase_UniprotID
0   41_654_666           P00533
1   41_654_666           P05771
2   41_654_666           Q9H3D0
3  ANXA1_14_26           P12931
4  ANXA1_14_26           P00533
5  ANXA1_14_26           P41743
6  ANXA1_14_26           Q96QT4
7  ANXA1_14_26           Q71UK5
8  ANXA1_14_26           Q9H3D0
9  ANXA1_14_26           Q13546


3. Comparision of the output of both dicts

In [30]:
def compare_peptide_kinase_mappings(peptide_to_kinases_KRSA, peptide_to_kinases_pyGamG):
    """
    Vergleicht zwei Peptid-zu-Kinase Mappings und gibt detaillierte Statistiken aus.
    """
    # Alle Peptide aus beiden Dictionaries
    all_peptides = set(peptide_to_kinases_KRSA.keys()) | set(peptide_to_kinases_pyGamG.keys())
    
    # Statistiken
    stats = {
        'total_peptides': len(all_peptides),
        'only_in_KRSA': 0,
        'only_in_pyGamG': 0,
        'in_both': 0,
        'identical_mappings': 0,
        'different_mappings': 0
    }
    
    comparison_results = []
    
    for peptide in sorted(all_peptides):
        kinases_KRSA = set(peptide_to_kinases_KRSA.get(peptide, []))
        kinases_pyGamG = set(peptide_to_kinases_pyGamG.get(peptide, []))
        
        # Überlappung berechnen
        in_both = kinases_KRSA & kinases_pyGamG
        only_KRSA = kinases_KRSA - kinases_pyGamG
        only_pyGamG = kinases_pyGamG - kinases_KRSA
        
        # Statistiken aktualisieren
        if peptide in peptide_to_kinases_KRSA and peptide not in peptide_to_kinases_pyGamG:
            stats['only_in_KRSA'] += 1
        elif peptide not in peptide_to_kinases_KRSA and peptide in peptide_to_kinases_pyGamG:
            stats['only_in_pyGamG'] += 1
        else:
            stats['in_both'] += 1
            if kinases_KRSA == kinases_pyGamG:
                stats['identical_mappings'] += 1
            else:
                stats['different_mappings'] += 1
        
        comparison_results.append({
            'peptide': peptide,
            'num_kinases_KRSA': len(kinases_KRSA),
            'num_kinases_pyGamG': len(kinases_pyGamG),
            'num_overlap': len(in_both),
            'only_KRSA': [k for k in only_KRSA if pd.notna(k)],  # NaN filtern
            'only_pyGamG': [k for k in only_pyGamG if pd.notna(k)],  # NaN filtern
            'overlap': [k for k in in_both if pd.notna(k)],  # NaN filtern
            'overlap_percent': len(in_both) / len(kinases_KRSA | kinases_pyGamG) * 100 if (kinases_KRSA | kinases_pyGamG) else 0
        })
    
    # DataFrame erstellen
    df_comparison = pd.DataFrame(comparison_results)
    
    # Statistiken ausgeben
    print("=" * 80)
    print("VERGLEICH: KRSA vs pyGamG Peptid-zu-Kinase Mappings")
    print("=" * 80)
    print(f"\nGesamtanzahl Peptide: {stats['total_peptides']}")
    print(f"  - Nur in KRSA: {stats['only_in_KRSA']} ({stats['only_in_KRSA']/stats['total_peptides']*100:.1f}%)")
    print(f"  - Nur in pyGamG: {stats['only_in_pyGamG']} ({stats['only_in_pyGamG']/stats['total_peptides']*100:.1f}%)")
    print(f"  - In beiden: {stats['in_both']} ({stats['in_both']/stats['total_peptides']*100:.1f}%)")
    
    if stats['in_both'] > 0:
        print(f"\nVon den {stats['in_both']} Peptiden in beiden:")
        print(f"  - Identische Kinase-Mappings: {stats['identical_mappings']} ({stats['identical_mappings']/stats['in_both']*100:.1f}%)")
        print(f"  - Unterschiedliche Mappings: {stats['different_mappings']} ({stats['different_mappings']/stats['in_both']*100:.1f}%)")
    
    print(f"\nDurchschnittliche Overlap: {df_comparison['overlap_percent'].mean():.1f}%")
    print(f"Median Overlap: {df_comparison['overlap_percent'].median():.1f}%")
    
    return df_comparison, stats

In [31]:
def compare_peptide_kinase_mappings(df_pairs_KRSA, df_pairs_pyGamG):
    """
    Vergleicht zwei Peptid-zu-Kinase Mappings und gibt detaillierte Statistiken aus.
    
    :param df_pairs_KRSA: DataFrame mit Spalten ['Peptide_ID', 'Kinase_UniprotID']
    :param df_pairs_pyGamG: DataFrame mit Spalten ['Peptide_ID', 'Kinase_UniprotID']
    :return: df_comparison, stats
    """
    # DataFrames zu Dictionaries konvertieren (Peptide -> Liste von Kinasen)
    peptide_to_kinases_KRSA = df_pairs_KRSA.groupby('Peptide_ID')['Kinase_UniprotID'].apply(list).to_dict()
    peptide_to_kinases_pyGamG = df_pairs_pyGamG.groupby('Peptide_ID')['Kinase_UniprotID'].apply(list).to_dict()
    
    # Alle Peptide aus beiden DataFrames
    all_peptides = set(peptide_to_kinases_KRSA.keys()) | set(peptide_to_kinases_pyGamG.keys())
    
    # Statistiken
    stats = {
        'total_peptides': len(all_peptides),
        'only_in_KRSA': 0,
        'only_in_pyGamG': 0,
        'in_both': 0,
        'identical_mappings': 0,
        'different_mappings': 0
    }
    
    comparison_results = []
    
    for peptide in sorted(all_peptides):
        kinases_KRSA = set(peptide_to_kinases_KRSA.get(peptide, []))
        kinases_pyGamG = set(peptide_to_kinases_pyGamG.get(peptide, []))
        
        # NaN-Werte entfernen
        kinases_KRSA = {k for k in kinases_KRSA if pd.notna(k)}
        kinases_pyGamG = {k for k in kinases_pyGamG if pd.notna(k)}
        
        # Überlappung berechnen
        in_both = kinases_KRSA & kinases_pyGamG
        only_KRSA = kinases_KRSA - kinases_pyGamG
        only_pyGamG = kinases_pyGamG - kinases_KRSA
        
        # Statistiken aktualisieren
        if peptide in peptide_to_kinases_KRSA and peptide not in peptide_to_kinases_pyGamG:
            stats['only_in_KRSA'] += 1
        elif peptide not in peptide_to_kinases_KRSA and peptide in peptide_to_kinases_pyGamG:
            stats['only_in_pyGamG'] += 1
        else:
            stats['in_both'] += 1
            if kinases_KRSA == kinases_pyGamG:
                stats['identical_mappings'] += 1
            else:
                stats['different_mappings'] += 1
        
        comparison_results.append({
            'peptide': peptide,
            'num_kinases_KRSA': len(kinases_KRSA),
            'num_kinases_pyGamG': len(kinases_pyGamG),
            'num_overlap': len(in_both),
            'only_KRSA': list(only_KRSA),
            'only_pyGamG': list(only_pyGamG),
            'overlap': list(in_both),
            'overlap_percent': len(in_both) / len(kinases_KRSA | kinases_pyGamG) * 100 if (kinases_KRSA | kinases_pyGamG) else 0
        })
    
    # DataFrame erstellen
    df_comparison = pd.DataFrame(comparison_results)
    
    # Statistiken ausgeben
    print("=" * 80)
    print("VERGLEICH: KRSA vs pyGamG Peptid-zu-Kinase Mappings")
    print("=" * 80)
    print(f"\nGesamtanzahl Peptide: {stats['total_peptides']}")
    print(f"  - Nur in KRSA: {stats['only_in_KRSA']} ({stats['only_in_KRSA']/stats['total_peptides']*100:.1f}%)")
    print(f"  - Nur in pyGamG: {stats['only_in_pyGamG']} ({stats['only_in_pyGamG']/stats['total_peptides']*100:.1f}%)")
    print(f"  - In beiden: {stats['in_both']} ({stats['in_both']/stats['total_peptides']*100:.1f}%)")
    
    if stats['in_both'] > 0:
        print(f"\nVon den {stats['in_both']} Peptiden in beiden:")
        print(f"  - Identische Kinase-Mappings: {stats['identical_mappings']} ({stats['identical_mappings']/stats['in_both']*100:.1f}%)")
        print(f"  - Unterschiedliche Mappings: {stats['different_mappings']} ({stats['different_mappings']/stats['in_both']*100:.1f}%)")
    
    print(f"\nDurchschnittliche Overlap: {df_comparison['overlap_percent'].mean():.1f}%")
    print(f"Median Overlap: {df_comparison['overlap_percent'].median():.1f}%")
    
    # Zusätzliche Statistik: Paar-Level Vergleich
    total_pairs_KRSA = len(df_pairs_KRSA)
    total_pairs_pyGamG = len(df_pairs_pyGamG)
    
    # Kombiniere beide DataFrames um gemeinsame Paare zu finden
    df_pairs_KRSA['source'] = 'KRSA'
    df_pairs_pyGamG['source'] = 'pyGamG'
    df_all_pairs = pd.concat([df_pairs_KRSA, df_pairs_pyGamG])
    
    # Paare die in beiden vorkommen
    pair_counts = df_all_pairs.groupby(['Peptide_ID', 'Kinase_UniprotID']).size()
    common_pairs = (pair_counts == 2).sum()
    
    print(f"\n--- Paar-Level Statistik ---")
    print(f"Total Paare KRSA: {total_pairs_KRSA}")
    print(f"Total Paare pyGamG: {total_pairs_pyGamG}")
    print(f"Gemeinsame Paare: {common_pairs}")
    print(f"Overlap auf Paar-Ebene: {common_pairs / max(total_pairs_KRSA, total_pairs_pyGamG) * 100:.1f}%")
    
    return df_comparison, stats

In [32]:
df_comparison, stats = compare_peptide_kinase_mappings(df_pairs_KRSA, df_pairs_pyGamG)

VERGLEICH: KRSA vs pyGamG Peptid-zu-Kinase Mappings

Gesamtanzahl Peptide: 339
  - Nur in KRSA: 83 (24.5%)
  - Nur in pyGamG: 72 (21.2%)
  - In beiden: 184 (54.3%)

Von den 184 Peptiden in beiden:
  - Identische Kinase-Mappings: 5 (2.7%)
  - Unterschiedliche Mappings: 179 (97.3%)

Durchschnittliche Overlap: 12.0%
Median Overlap: 0.0%

--- Paar-Level Statistik ---
Total Paare KRSA: 1212
Total Paare pyGamG: 4851
Gemeinsame Paare: 580
Overlap auf Paar-Ebene: 12.0%


4. Convert KRSA mapping for direct use (WITHOUT BLAST/PTM lookup)

In [33]:
def convert_KRSA_to_kinase_peptide_dict(df_pairs_KRSA, df_pooled, output_path=None):
    """
    Konvertiert KRSA Peptid-Kinase Paare direkt in das kinase_to_peptides Format.
    
    WICHTIG: Diese Funktion überspringt BLAST und PTM Lookups komplett!
    KRSA enthält bereits fertige Kinase→Peptid Mappings.
    
    :param df_pairs_KRSA: DataFrame mit ['Peptide_ID', 'Kinase_UniprotID']
    :param df_pooled: DataFrame mit Peptid-Statistiken (für mean_control/mean_treatment)
    :param output_path: Optional - Pfad zum Speichern als JSON
    :return: Dictionary {kinase: [{peptide_id, mean_control, mean_treatment}, ...]}
    """
    from collections import defaultdict
    
    # Erstelle Kinase -> Liste von Peptiden Dictionary
    kinase_to_peptides = defaultdict(list)
    kinase_peptide_seen = defaultdict(set)
    
    # Erstelle lookup für Peptid-Statistiken
    peptide_stats = {}
    for _, row in df_pooled.iterrows():
        peptide_stats[row['ID']] = {
            'mean_control': row.get('mean_control', 0),
            'mean_treatment': row.get('mean_treatment', 0)
        }
    
    # Durchlaufe KRSA Mappings und baue kinase_to_peptides auf
    for _, row in df_pairs_KRSA.iterrows():
        peptide_id = row['Peptide_ID']
        kinase = row['Kinase_UniprotID']
        
        # Überspringe NaN-Werte
        if pd.isna(kinase) or pd.isna(peptide_id):
            continue
        
        # Nur neue Peptid-Kinase Paare hinzufügen
        if peptide_id not in kinase_peptide_seen[kinase]:
            # Hole Peptid-Statistiken falls vorhanden
            stats = peptide_stats.get(peptide_id, {'mean_control': 0, 'mean_treatment': 0})
            
            kinase_to_peptides[kinase].append({
                'peptide_id': peptide_id,
                'mean_control': stats['mean_control'],
                'mean_treatment': stats['mean_treatment']
            })
            kinase_peptide_seen[kinase].add(peptide_id)
    
    print(f"KRSA Direct Mapping Statistics:")
    print(f"  Input KRSA pairs: {len(df_pairs_KRSA)}")
    print(f"  Unique kinases: {len(kinase_to_peptides)}")
    print(f"  Total kinase-peptide connections: {sum(len(peptides) for peptides in kinase_to_peptides.values())}")
    
    # Optional: Als JSON speichern
    if output_path:
        import json
        with open(output_path, 'w') as f:
            json.dump(dict(kinase_to_peptides), f, indent=2)
        print(f"\nSaved to: {output_path}")
    
    return dict(kinase_to_peptides)

In [34]:
# Konvertiere KRSA Mappings direkt (ohne BLAST/PTM)
output_path = Path('../data/benchmarking/testing_kinase_substrate_mapping/KRSA_kinase_to_peptides.json')

kinase_to_peptides_KRSA = convert_KRSA_to_kinase_peptide_dict(
    df_pairs_KRSA=df_pairs_KRSA,
    df_pooled=df_peptide_enrichment,
    output_path=output_path
)

# Zeige Beispiel
if kinase_to_peptides_KRSA:
    example_kinase = list(kinase_to_peptides_KRSA.keys())[0]
    print(f"\nBeispiel: Kinase {example_kinase}")
    print(f"  Anzahl gemappte Peptide: {len(kinase_to_peptides_KRSA[example_kinase])}")
    print(f"  Erste 3 Peptide: {kinase_to_peptides_KRSA[example_kinase][:3]}")

KRSA Direct Mapping Statistics:
  Input KRSA pairs: 1212
  Unique kinases: 210
  Total kinase-peptide connections: 1212

Saved to: ../data/benchmarking/testing_kinase_substrate_mapping/KRSA_kinase_to_peptides.json

Beispiel: Kinase P00533
  Anzahl gemappte Peptide: 20
  Erste 3 Peptide: [{'peptide_id': '41_654_666', 'mean_control': 0, 'mean_treatment': 0}, {'peptide_id': 'ANXA1_14_26', 'mean_control': 0, 'mean_treatment': 0}, {'peptide_id': 'CBL_693_705', 'mean_control': 0, 'mean_treatment': 0}]


5. Speichere auch als CSV für einfaches Laden in kx_UKA_KSEA.py

In [39]:
# Speichere KRSA Mappings auch als einfache CSV
# Format: Peptide_ID, Kinase_UniprotID (ohne BLAST/PTM Umwandlung)

output_csv = Path('../data/benchmarking/testing_kinase_substrate_mapping/KRSA_peptide_kinase_mapping.csv')

# Entferne NaN-Werte und speichere
df_pairs_KRSA_clean = df_pairs_KRSA.dropna()
#df_pairs_KRSA_clean.to_csv(output_csv, index=False)

#print(f"Saved clean KRSA mapping to: {output_csv}")
print(f"Rows: {len(df_pairs_KRSA_clean)}")
print(f"\nFormat: Peptide_ID, Kinase_UniprotID")
print(df_pairs_KRSA_clean.head(n=25))

Rows: 1212

Format: Peptide_ID, Kinase_UniprotID
            Peptide_ID Kinase_UniprotID
0           41_654_666           P00533
3         AKT1_170_182           Q07912
4   AKT1_309_321_C310S           P07949
5   AKT1_309_321_C310S           P12931
6   AKT1_309_321_C310S           P31749
7   AKT1_309_321_C310S           Q13882
8   AKT1_309_321_C310S           P06239
9   AKT1_309_321_C310S           P00519
11        AKT1_320_332           P12931
12        AKT1_320_332           P31749
13        AKT1_320_332           Q13882
14         ANXA1_14_26           P00519
16         ANXA1_14_26           P00533
19         ANXA1_14_26           P12931
22         ANXA2_17_29           P08069
23         ANXA2_17_29           P12931
27        ARAF_297_307           P12931
31        ARAF_297_307           O60674
40         CBL_693_705           P00519
41         CBL_693_705           P00533
42         CBL_693_705           P06213
43         CBL_693_705           P06239
44         CBL_693_705         